# RQ1 — Rail LU TWT: Candidate K Evaluation

Loads the pre-built feature matrix and diagnostics from Notebook 1 and
produces visual and tabular comparisons for the candidate K values
(default: K=3 and K=7, reflecting the metric split).

**Prerequisite:** run Notebook 1 first to generate the parquet + CSV inputs.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
from pathlib import Path

# Container (Podman) path to the FYP project root — the folder that holds
# PROJECT_CONTEXT.md and outputs/.  In-container path, NOT a Windows drive letter.
#   - Set to None to auto-discover by walking up from the kernel cwd.
PROJECT_ROOT_RELATIVE = Path("/home/jovyan/work/CASA_FYP/FYP")

CANDIDATE_KS = [3, 7]   # adjust after inspecting Notebook 1 diagnostics
RANDOM_STATE = 42

print("Kernel working directory:", Path.cwd())
print("Configured project root :", PROJECT_ROOT_RELATIVE)

## 1  Imports

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

%matplotlib inline

## 2  Path helpers

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    if start is None:
        try:
            start = Path(__file__).resolve()
        except NameError:
            start = Path.cwd().resolve()
    if start.is_file():
        start = start.parent
    for candidate in [start, *start.parents]:
        if (candidate / "PROJECT_CONTEXT.md").exists():
            return candidate
    raise FileNotFoundError(
        "PROJECT_CONTEXT.md not found. Set PROJECT_ROOT_RELATIVE in the Config cell."
    )


def resolve_project_root(rel: Path | None = None) -> Path:
    if rel is not None:
        # absolute path: use directly; relative path: resolve from cwd
        candidate = rel.resolve() if rel.is_absolute() else (Path.cwd() / rel).resolve()
        if not (candidate / "PROJECT_CONTEXT.md").exists():
            raise FileNotFoundError(
                f"PROJECT_ROOT_RELATIVE resolved to {candidate} "
                "but PROJECT_CONTEXT.md was not found there."
            )
        return candidate
    return find_project_root()


ROOT      = resolve_project_root(PROJECT_ROOT_RELATIVE)
INPUT_DIR = ROOT / "outputs" / "rq1_trial_numbat_lu_only"
OUTPUT_DIR = INPUT_DIR / "candidate_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_PATH             = INPUT_DIR / "X_rail_LU_TWT_trial.parquet"
META_PATH          = INPUT_DIR / "rail_LU_TWT_meta_trial.csv"
GMM_DIAG_PATH      = INPUT_DIR / "rail_LU_TWT_gmm_k_diagnostics.csv"
KMEANS_DIAG_PATH   = INPUT_DIR / "rail_LU_TWT_kmeans_k_diagnostics.csv"

print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)

## 3  Load data

In [ ]:
X          = pd.read_parquet(X_PATH)
X.index    = X.index.astype(str)
meta       = pd.read_csv(META_PATH, dtype={"NLC": "string", "ASC": "string"})
meta["NLC"] = meta["NLC"].astype(str)
meta       = meta.set_index("NLC").loc[X.index].reset_index()
gmm_diag   = pd.read_csv(GMM_DIAG_PATH)
kmeans_diag = pd.read_csv(KMEANS_DIAG_PATH)

print(f"X shape      : {X.shape}")
print(f"meta shape   : {meta.shape}")
print(f"Candidate Ks : {CANDIDATE_KS}")
display(meta.head())

## 4  K diagnostics plot

In [ ]:
def plot_k_diagnostics(gmm_diag: pd.DataFrame, kmeans_diag: pd.DataFrame,
                       candidate_ks: list[int]) -> None:
    specs = [
        ("silhouette",        "Silhouette score",        "higher is better"),
        ("calinski_harabasz", "Calinski-Harabasz index", "higher is better"),
        ("davies_bouldin",    "Davies-Bouldin index",    "lower is better"),
    ]
    fig, axes = plt.subplots(3, 1, figsize=(9, 9), sharex=True)
    for ax, (col, title, note) in zip(axes, specs):
        ax.plot(gmm_diag["k"],    gmm_diag[col],    marker="o", label="GMM")
        ax.plot(kmeans_diag["k"], kmeans_diag[col], marker="s", label="KMeans")
        for k in candidate_ks:
            ax.axvline(k, color="#999999", linestyle="--", linewidth=0.9,
                       label=f"K={k}" if ax is axes[0] else None)
        ax.set_ylabel(title)
        ax.set_title(f"{title} ({note})")
        ax.grid(color="#eeeeee", linewidth=0.8)
        ax.spines[["top","right"]].set_visible(False)
    axes[-1].set_xlabel("Number of clusters (K)")
    axes[0].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "k_diagnostics_gmm_kmeans.png", dpi=200)
    plt.show()


plot_k_diagnostics(gmm_diag, kmeans_diag, CANDIDATE_KS)

## 5  Helper functions for candidate evaluation

In [ ]:
def cluster_sizes(labels: np.ndarray) -> dict:
    vals, cnts = np.unique(labels, return_counts=True)
    return {str(int(v)): int(c) for v, c in zip(vals, cnts)}


def posterior_entropy(posterior: np.ndarray) -> np.ndarray:
    clipped = np.clip(posterior, 1e-12, 1.0)
    return -(posterior * np.log(clipped)).sum(axis=1)


def fit_gmm(X: pd.DataFrame, k: int):
    model  = GaussianMixture(n_components=k, covariance_type="full",
                              n_init=50, tol=1e-6, max_iter=300,
                              random_state=RANDOM_STATE)
    labels    = model.fit_predict(X)
    posterior = model.predict_proba(X)
    return model, labels, posterior


def bin_start_minutes(bin_label: str) -> int:
    h, m  = int(bin_label[:2]), int(bin_label[2:4])
    total = h * 60 + m
    return total + 24 * 60 if total < 18 * 60 else total


def format_night_axis(ax) -> None:
    ax.set_xlim(18 * 60, 25 * 60)
    ax.set_xticks([18*60, 21*60, 24*60, 25*60])
    ax.set_xticklabels(["18:00", "21:00", "00:00", "01:00"])
    ax.grid(axis="y", color="#dddddd", linewidth=0.8)
    ax.spines[["top","right"]].set_visible(False)

## 6  Evaluate each candidate K

In [ ]:
def parse_feature_table(X: pd.DataFrame, labels: np.ndarray) -> pd.DataFrame:
    rows = []
    for nlc, label in zip(X.index.astype(str), labels):
        for feature, value in X.loc[nlc].items():
            day_type, direction, bin_label = feature.split("_", 2)
            rows.append({"NLC": nlc, "cluster": int(label),
                         "day_type": day_type, "direction": direction,
                         "bin_label": bin_label, "share": float(value)})
    return pd.DataFrame(rows)


def plot_cluster_profiles(X: pd.DataFrame, labels: np.ndarray, k: int) -> pd.DataFrame:
    long_df = parse_feature_table(X, labels)
    long_df["plot_minute"] = long_df["bin_label"].apply(bin_start_minutes)

    profile = (
        long_df.groupby(["cluster","direction","bin_label","plot_minute"], as_index=False)
        .agg(median_share=("share","median"), mean_share=("share","mean"),
             p10_share=("share", lambda v: v.quantile(0.10)),
             p90_share=("share", lambda v: v.quantile(0.90)))
        .sort_values(["cluster","direction","plot_minute"])
    )
    profile.to_csv(OUTPUT_DIR / f"k{k}_cluster_temporal_profiles.csv", index=False)

    clusters = sorted(profile["cluster"].unique())
    fig, axes = plt.subplots(len(clusters), 1,
                             figsize=(10, max(2.6*len(clusters), 4)),
                             sharex=True, sharey=False)
    if len(clusters) == 1:
        axes = [axes]

    for ax, cluster in zip(axes, clusters):
        subset = profile[profile["cluster"] == cluster]
        for direction, color in [("entry","#2F6B4F"), ("exit","#9A3D3D")]:
            line = subset[subset["direction"] == direction].sort_values("plot_minute")
            ax.plot(line["plot_minute"], line["median_share"],
                    marker="o", markersize=3, linewidth=1.6, color=color, label=direction)
            ax.fill_between(line["plot_minute"].to_numpy(float),
                            line["p10_share"].to_numpy(float),
                            line["p90_share"].to_numpy(float),
                            color=color, alpha=0.12, linewidth=0)
        n_stations = (labels == cluster).sum()
        ax.set_title(f"K={k}  Cluster {cluster}  (n={n_stations})")
        ax.set_ylabel("Feature share")
        format_night_axis(ax)

    axes[0].legend(loc="upper right")
    axes[-1].set_xlabel("Time")
    fig.suptitle(f"GMM K={k} — Entry/Exit median temporal profiles (TWT, 18:00–01:00)",
                 fontsize=11, y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"k{k}_entry_exit_temporal_profiles.png", dpi=200,
                bbox_inches="tight")
    plt.show()
    return profile


def build_cluster_summary(X: pd.DataFrame, meta: pd.DataFrame,
                           model, labels: np.ndarray, posterior: np.ndarray,
                           k: int) -> pd.DataFrame:
    entropy        = posterior_entropy(posterior)
    max_post       = posterior.max(axis=1)
    label_df       = meta.copy()
    label_df["cluster"]           = labels
    label_df["max_posterior"]     = max_post
    label_df["posterior_entropy"] = entropy
    label_df.to_csv(OUTPUT_DIR / f"k{k}_labels_with_posterior.csv", index=False)

    summary = (
        label_df.groupby("cluster", as_index=False)
        .agg(station_count=("NLC","count"),
             total_activity_sum=("total_rail_activity","sum"),
             total_activity_median=("total_rail_activity","median"),
             mean_max_posterior=("max_posterior","mean"),
             min_max_posterior=("max_posterior","min"),
             mean_entropy=("posterior_entropy","mean"))
        .sort_values("cluster")
    )
    summary.to_csv(OUTPUT_DIR / f"k{k}_cluster_summary.csv", index=False)

    model_meta = {"k": k, "bic": float(model.bic(X)), "aic": float(model.aic(X)),
                  "log_likelihood": float(model.score(X)),
                  "converged": bool(model.converged_), "n_iter": int(model.n_iter_),
                  "cluster_sizes": cluster_sizes(labels),
                  "mean_max_posterior": float(max_post.mean()),
                  "min_max_posterior": float(max_post.min()),
                  "mean_entropy": float(entropy.mean())}
    with open(OUTPUT_DIR / f"k{k}_model_summary.json", "w", encoding="utf-8") as f:
        json.dump(model_meta, f, indent=2)
    return summary

In [ ]:
labels_by_k     = {}
comparison_rows = []

for k in CANDIDATE_KS:
    print(f"\n{'='*60}")
    print(f"  Fitting GMM  K = {k}")
    print(f"{'='*60}")

    model, labels, posterior = fit_gmm(X, k)
    labels_by_k[k] = labels

    summary = build_cluster_summary(X, meta, model, labels, posterior, k)
    print(f"\nCluster summary (K={k}):")
    display(summary)

    profile = plot_cluster_profiles(X, labels, k)

    entropy = posterior_entropy(posterior)
    comparison_rows.append({
        "k": k,
        "bic":                model.bic(X),
        "aic":                model.aic(X),
        "log_likelihood":     model.score(X),
        "converged":          model.converged_,
        "n_iter":             model.n_iter_,
        "mean_max_posterior": posterior.max(axis=1).mean(),
        "min_max_posterior":  posterior.max(axis=1).min(),
        "mean_entropy":       entropy.mean(),
        "cluster_sizes":      json.dumps(cluster_sizes(labels)),
    })

## 7  PCA visualisation

In [ ]:
def plot_pca(X: pd.DataFrame, labels_by_k: dict) -> pd.DataFrame:
    pca    = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X)

    fig, axes = plt.subplots(1, len(labels_by_k),
                             figsize=(7 * len(labels_by_k), 6))
    if len(labels_by_k) == 1:
        axes = [axes]

    for ax, (k, labels) in zip(axes, labels_by_k.items()):
        sc = ax.scatter(coords[:,0], coords[:,1], c=labels, cmap="tab10",
                        s=32, alpha=0.85, edgecolor="white", linewidth=0.3)
        ax.set_title(f"PCA — LU TWT profiles, K={k}")
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
        ax.grid(color="#eeeeee", linewidth=0.8)
        ax.spines[["top","right"]].set_visible(False)
        ax.legend(*sc.legend_elements(), title="Cluster")

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "candidate_pca_clusters.png", dpi=200)
    plt.show()

    pca_table = pd.DataFrame({"NLC": X.index.astype(str),
                               "pc1": coords[:,0], "pc2": coords[:,1]})
    for k, labels in labels_by_k.items():
        pca_table[f"k{k}_cluster"] = labels
    pca_table.to_csv(OUTPUT_DIR / "candidate_pca_coordinates.csv", index=False)
    return pca_table


pca_table = plot_pca(X, labels_by_k)
display(pca_table.head())

## 8  Cross-candidate comparison

In [ ]:
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(OUTPUT_DIR / "candidate_model_comparison.csv", index=False)

print("Cross-candidate comparison:")
display(comparison[["k","bic","aic","log_likelihood","converged","n_iter",
                     "mean_max_posterior","min_max_posterior","mean_entropy"]])
print("\nAll outputs saved to:", OUTPUT_DIR)